## Feature Scaling

Feature scaling puts all numeric features on a comparable scale so no feature dominates unfairly.

### Why scaling exists

Many models rely on:

- distance

- gradients

- optimization speed

If one feature has large numbers, it overpowers others.

Already saw this with KNN.

### Models that REQUIRE scaling (non-negotiable)

| Model               | Why                   |
| ------------------- | --------------------- |
| KNN                 | Distance-based        |
| Logistic Regression | Gradient optimization |
| SVM                 | Margin calculation    |
| PCA                 | Variance-based        |


### Models that DO NOT NEED scaling

| Model             | Why               |
| ----------------- | ----------------- |
| Decision Trees    | Rule-based splits |
| Random Forest     | Tree-based        |
| Gradient Boosting | Tree-based        |

Scaling won’t help — but also won’t break them.


### Types of scalers

#### 🔹 StandardScaler (Standardization)

- Most Common scaler

Formula:

x' = $\frac{x-mean}{std}$

- Mean → 0

- Std → 1

- Best for most ML models

What it does
- subtracts the mean
- divides by standard deviation

 Result
- mean becomes 0
- std becomes 1
- values usually fall between -3 and +3 (not guaranteed)

 Properties
- keeps relative distances
- sensitive to outliers

 Best used for
- Logistic Regression
- KNN
- SVM
- PCA

#### 🔹 MinMaxScaler (Normalization)

Formula:

x' = $\frac{x-min}{max-min}$

- Range → [0, 1]

- Sensitive to outliers


 What it does
- rescales data into a fixed range

 result
- values lie in [0, 1] (default)

 properties
- preserves shape of distribution
- very sensitive to outliers

 best used for
- neural networks
- when a bounded range is required

#### 🔹 RobustScaler

Formula (conceptual):

$$
x' = \frac{x - \text{median}}{\text{IQR}}
$$


Uses median and IQR

Handles outliers well

What it uses
- median instead of mean
- IQR instead of standard deviation

Result
- ignores extreme outliers
- values are not strictly bounded

Best used for
- datasets with heavy outliers

### Which scaler should YOU default to?

Default choice → StandardScaler

Change only if:

- Data has heavy outliers → RobustScaler

- Need bounded range → MinMaxScaler

### CRITICAL rule (never break this)

(fitting data)
```
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train) 
X_test_scaled  = scaler.transform(X_test) 

❌ Never fit on test data <br>
❌ Never scale before splitting

Why? → Data leakage

| Question                          | Answer |
| --------------------------------- | ------ |
| Does scaling change data meaning? | ❌ No   |
| Does it change distances?         | ✅ Yes  |
| Mandatory for KNN?                | ✅      |
| Mandatory for trees?              | ❌      |


## Encoding Categorical Data

How do we convert text into numbers without lying to the model?

### What is categorical data?

Categorical data represents labels, not quantities.

Examples:
- sex → Male / Female
- smoker → Yes / No
- day → Thur / Fri / Sat / Sun

Models cannot work with strings.
They only understand numbers.

### Why encoding is dangerous if done wrong

If you encode categories incorrectly, the model may assume:
- order where none exists
- distance where none exists

This leads to silent logical errors (very common).

### Label Encoding

Label Encoding assigns a unique number to each category.

Example:

- Male   → 0
- Female → 1

This is done using:

In [ ]:
from sklearn.preprocessing import LabelEncoder

#### When Label Encoding is SAFE

- binary categories (Yes / No)
- ordinal data (Low < Medium < High)

Example (safe):

- Low    → 0
- Medium → 1
- High   → 2


#### When Label Encoding is DANGEROUS

For nominal categories with no order.

Example:

- Red   → 0
- Blue  → 1
- Green → 2

The model may assume:

Green > Blue > Red

This is wrong, but the model doesn’t know that.

### One-Hot Encoding

Instead of one column with many values (1, 2, 3), One-Hot Encoding creates multiple columns, each for one value (in this case a day).

Example (day):

original data: day = Thurs


The model will now check and fill each column

- Is it Thur? → yes (1)
- Is it Fri? → no (0)
- Is it Sat? → no (0)
- Is it Sun? → no (0)

| **day_Thur** | day_Fri | day_Sat | day_Sun |
|-|-|-|-|
|   1     |    0    |    0      |  0 |

Each column answers:

is this category present?

- No order
- No magnitude
- No distance

This is the safest encoding for nominal data.

In [1]:
from sklearn.preprocessing import OneHotEncoder

### Dummy Variable Trap (intuition only)

If you have k categories, One-Hot Encoding creates k columns.

But there is no need for the last column, we can just assume the value.

Example:
- if `day_Thur = 0`, `day_Fri = 0`, `day_Sat = 0`
- then `day_Sun` must be `1`

To avoid redundancy:
- drop one column

Scikit-learn handles this automatically when needed.

### Which encoding should YOU use?

- binary categories → Label Encoding
- ordered categories → Label Encoding
- unordered categories → One-Hot Encoding

Default rule:
- if unsure → One-Hot Encode


### Key mistakes to avoid

- never Label Encode unordered categories
- never encode before train/test split
- never assume numbers imply meaning

## Handling Missing Values

What do we do when data is incomplete?

### What are missing values?

Missing values mean:
- the value exists conceptually
- but is not recorded

In pandas, missing values appear as:
- NaN
- None

|age|
|----|
|23|
|NaN|
|35|


### Why missing values are a problem

Most ML models:
- cannot train with missing values
- will throw an error or behave incorrectly

So we must handle them before training.

### The worst beginner mistake ❌

Dropping rows blindly:

df.dropna()

Why this is bad:
- you lose data
- you may remove important patterns
- results become biased

Dropping should be a last resort, not default.

### Main ways to handle missing values

#### 1️. Drop missing values (RARE)

This is acceptable only when:
- very few rows are missing
- dataset is large
- missingness is random

Example:

df.dropna()


Use this only if you are sure.

#### 2️. Fill missing values (MOST COMMON)

We replace missing values with a reasonable estimate.

##### Mean imputation

$$
x_{missing} = \text{mean of column}
$$

Use when:
- data is roughly symmetric
- no strong outliers

##### Median imputation (BETTER DEFAULT)

$$
x_{missing} = \text{median of column}
$$


Use when:
- data has outliers
- distribution is skewed



##### Mode imputation (for categorical data)

$$
x_{missing} = \text{most frequent value}
$$


Use when:
- feature is categorical



#### 3️. Add a “missing” indicator (IMPORTANT IDEA)

Sometimes:
- missing itself is meaningful

Example:
- income missing → user didn’t want to disclose

In this case:
- fill missing values
- add a new column indicating missingness

Example:

income_missing <br>
0 = value present <br>
1 = value was missing <br>


This lets the model learn from missingness itself.

### CRITICAL rule (NEVER BREAK THIS)

Compute fill values only on training data.

```
fill_value = X_train["age"].median()
X_train["age"] = X_train["age"].fillna(fill_value)
X_test["age"]  = X_test["age"].fillna(fill_value)
```


#### Step 1: Look ONLY at training data

Let’s say you have missing values in column age.

You compute the fill value from training data only:

fill_value = median(age in training data)


Why?
- because training data represents “what the model knows”
- test data must remain “unseen”

#### Step 2: Apply the SAME fill to both sets

training missing values → filled using fill_value
- test missing values → filled using the SAME fill_value

So both datasets are numerically consistent.

Why:
- prevents data leakage
- keeps evaluation honest

### What NOT to do ❌

❌ Wrong approach (data leakage)

- compute median using full dataset
- or compute median using test data

Why this is wrong:
- test data influences preprocessing
- model indirectly “sees the future”
- evaluation becomes optimistic (fake accuracy)

Dont's:
- never fill using test data statistics
- never ignore missing values
- never assume missing means zero

## Feature Transformation

Why does the same model behave badly on some numeric columns even after scaling?

### What is feature transformation?

Feature transformation means changing the shape of the data, not just its scale.

Scaling changes range. <br>
Transformation changes distribution.

### The real problem: skewed data

Skewed data is an asymmetrical distribution where data points cluster on one side, creating a long "tail" on the other, unlike a symmetrical bell curve (normal distribution). 

Many real-world numeric features are not symmetric.

Example:
- income
- house prices
- transaction amounts

Most values are small, a few are very large.

This is called right-skewed data.

<img src="Images/skewed data.png">

### Why skewed data is a problem

Models like:
- Linear Regression
- Logistic Regression

assume:
- relationships are roughly linear
- errors are evenly distributed

Skewed data violates these assumptions.

Result:
- unstable coefficients
- poor generalization

### Log transformation (MOST IMPORTANT)

Log transformation compresses large values and spreads small ones.

Formula:

$$
x' = \log(x)
$$


If zero values exist:

$$
x' = \log(x + 1)
$$

#### What log transformation actually does

Suppose you have:

| value |
|-------|
| 10    |
| 100   |
| 1000  |


After log transform (conceptually):

| log(value) |
|------------|
| 1          |
| 2          |
| 3          |


Large gaps become manageable. <br>
The distribution becomes more symmetric.

#### When should you use log transformation?

- feature has extreme outliers
- feature is right-skewed
- model struggles even after scaling

Common examples:
- total_bill
- income
- price

#### When you should NOT use it

- data is already symmetric
- data contains negative values (log is undefined)
- tree-based models (trees don’t care about distribution shape)

### Square root transformation (alternative)

Sometimes a milder transformation is enough.

Formula:

$$
x' = \sqrt{x}
$$


Used when:
- skew exists but is not extreme

#### Very important rule (NEVER BREAK)

Transformations must follow the same rule as scaling and filling:

- learn transformation logic from training data
- apply the SAME transformation to test data

Never “peek” at test data to decide transformations.

#### Common beginner mistake ❌

Applying transformation blindly:

- log-transforming every numeric column
- transforming target variable without understanding meaning

Transformation is a tool, not a default step.

## Data Leakage

Why does a model look amazing in training/testing but fail in real life?

### What is data leakage?

Data leakage happens when information from the future (test data) accidentally influences training or preprocessing.

The model then:
- learns things it should not know
- gives unrealistically high accuracy
- fails badly on new data

This is the most dangerous ML mistake because it looks correct.

Test data must behave like unseen future data.

If the model “sees” it in any way:
- results are invalid

### Common ways data leakage happens

#### 1️. Scaling before train/test split

Wrong:

`scaler.fit(X)`   # uses full dataset ❌


Why this is leakage:
- test data affects mean and std
- model indirectly learns test distribution

Correct:

`scaler.fit(X_train)`


#### 2️. Filling missing values using full data

Wrong:

`df["age"].fillna(df["age"].median())`


Why:
- test data influences fill value

Correct:
- compute fill value from training data
- apply the same value to test data

#### 3️. Encoding before split 

Wrong:
- One-Hot Encoding entire dataset first
- then splitting

Why:
- category frequency from test data leaks

Correct:
- split first
- encode using training data rules

#### 4️. Feature creation using target information (VERY COMMON)

Example:
- creating a feature using future sales
- calculating averages using full timeline

This leaks answers into inputs.

### Why leakage gives “too good” results

Because:
- model sees patterns that won’t exist in reality
- evaluation no longer represents deployment

High accuracy + leakage = false confidence.

## Pipelines

### What is a pipeline?

A pipeline is a fixed sequence of steps that always runs in the same order.

Conceptually:

- preprocess data
- then train model
- then predict

All as one unit.

You no longer manually call:
- fill
- scale
- encode
- model

The pipeline does it safely and automatically.

### Why pipelines exist

Manual preprocessing causes:
- data leakage
- inconsistent train/test handling
- forgotten steps during prediction

Pipelines solve this by enforcing the rule:

Learn preprocessing only from training data, then reuse it unchanged.

### What a pipeline looks like conceptually

raw data <br>
→ missing value handling <br> 
→ encoding <br>
→ scaling <br>
→ model <br>

You define this once.

After that:
- .fit() handles training correctly
- .predict() applies the same steps automatically

### Why pipelines are a big deal

Without pipelines:
- you must remember the correct order
- one mistake ruins evaluation

With pipelines:
- order is guaranteed
- leakage is prevented
- deployment becomes realistic

This is why pipelines are professional ML, not optional sugar.

### Minimal pipeline example (classification)

In [21]:
from sklearn.model_selection import train_test_split
import seaborn as sns

df = sns.load_dataset("tips")
X = df[["tip", "size", "total_bill"]]
y = df["smoker"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [22]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

This means:

- step 1: scale features
- step 2: train logistic regression

### Training with a pipeline

In [23]:
pipe.fit(X_train, y_train)

,steps,"[('scaler', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0


What happens internally:

- scaler learns mean/std from X_train
- model trains on scaled X_train

### Prediction with a pipeline

In [24]:
y_pred = pipe.predict(X_test)

What happens internally:

- X_test is scaled using training statistics
- model predicts

You cannot accidentally leak data anymore.

### Pipelines + evaluation

In [25]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

          No       0.66      0.94      0.77        31
         Yes       0.60      0.17      0.26        18

    accuracy                           0.65        49
   macro avg       0.63      0.55      0.52        49
weighted avg       0.64      0.65      0.59        49



### Why pipelines matter even more later

Pipelines become critical when you add:

- One-Hot Encoding
- missing value imputers
- feature transformations
- hyperparameter tuning
- cross-validation

All of those break easily without pipelines.